# Machine Learning Preprocessing 

## Import libraries 

In [ ]:
import pandas as pd 

## Load somatic cancer variants

In [ ]:
variants = pd.read_csv(
  "/home/anekl/git/master/cancer_variants_annotation_pipeline/output/variants_with_maves.tsv",
  sep="\t", 
  low_memory=False
)

print(f"Loaded {len(variants)} variants.")

Loaded 923011 variants.


In [ ]:
print("The variant data contains the following columns:")
print(variants.columns.tolist())

The variant data contains the following columns:
['Hugo_Symbol', 'Entrez_Gene_Id', 'Center', 'NCBI_Build', 'Chromosome', 'Start_Position', 'End_Position', 'Strand', 'Consequence', 'Variant_Classification', 'Variant_Type', 'Reference_Allele', 'Tumor_Seq_Allele1', 'Tumor_Seq_Allele2', 'dbSNP_RS', 'dbSNP_Val_Status', 'Tumor_Sample_Barcode', 'Matched_Norm_Sample_Barcode', 'Match_Norm_Seq_Allele1', 'Match_Norm_Seq_Allele2', 'Tumor_Validation_Allele1', 'Tumor_Validation_Allele2', 'Match_Norm_Validation_Allele1', 'Match_Norm_Validation_Allele2', 'Verification_Status', 'Validation_Status', 'Mutation_Status', 'Sequencing_Phase', 'Sequence_Source', 'Validation_Method', 'Score', 'BAM_File', 'Sequencer', 't_ref_count', 't_alt_count', 'n_ref_count', 'n_alt_count', 'HGVSc', 'HGVSp', 'HGVSp_Short', 'Transcript_ID', 'RefSeq', 'Protein_position', 'Codons', 'Exon_Number', 'gnomAD_AF', 'gnomAD_AFR_AF', 'gnomAD_AMR_AF', 'gnomAD_ASJ_AF', 'gnomAD_EAS_AF', 'gnomAD_FIN_AF', 'gnomAD_NFE_AF', 'gnomAD_OTH_AF', '

## Create new columns 

In [ ]:
# Column 'n_func_sites' for variants with several functional sites 
variants['n_func_sites'] = variants['FEATURE_TYPE'].apply(
  lambda x: len(str(x).split(';')) if pd.notna(x) and str(x).strip() != "" else 0
)

print(variants[['n_func_sites', 'FEATURE_TYPE']].head(3))

   n_func_sites  FEATURE_TYPE
0             0           NaN
1             0           NaN
2             1  Binding site


In [ ]:
# Column 'n_protein_domains' for variants with several protein domains 
variants['n_protein_domains'] = variants['DOMAIN_NAME'].apply(
  lambda x: len(str(x).split(';')) if pd.notna(x) and str(x).strip() != '' else 0
)

print(variants[['DOMAIN_NAME', 'n_protein_domains']].head(3))

  DOMAIN_NAME  n_protein_domains
0     FERM_F1                  1
1     Pkinase                  1
2     Pkinase                  1


In [ ]:
# Boolean column (True/False) to check if a variant has a gnomAD allele frequency 
variants['has_gnomAD_AF'] = variants['gnomAD_AF'].notna() 

## Choose features for ML model

In [ ]:
print(variants.columns.tolist())

['Hugo_Symbol', 'Entrez_Gene_Id', 'Center', 'NCBI_Build', 'Chromosome', 'Start_Position', 'End_Position', 'Strand', 'Consequence', 'Variant_Classification', 'Variant_Type', 'Reference_Allele', 'Tumor_Seq_Allele1', 'Tumor_Seq_Allele2', 'dbSNP_RS', 'dbSNP_Val_Status', 'Tumor_Sample_Barcode', 'Matched_Norm_Sample_Barcode', 'Match_Norm_Seq_Allele1', 'Match_Norm_Seq_Allele2', 'Tumor_Validation_Allele1', 'Tumor_Validation_Allele2', 'Match_Norm_Validation_Allele1', 'Match_Norm_Validation_Allele2', 'Verification_Status', 'Validation_Status', 'Mutation_Status', 'Sequencing_Phase', 'Sequence_Source', 'Validation_Method', 'Score', 'BAM_File', 'Sequencer', 't_ref_count', 't_alt_count', 'n_ref_count', 'n_alt_count', 'HGVSc', 'HGVSp', 'HGVSp_Short', 'Transcript_ID', 'RefSeq', 'Protein_position', 'Codons', 'Exon_Number', 'gnomAD_AF', 'gnomAD_AFR_AF', 'gnomAD_AMR_AF', 'gnomAD_ASJ_AF', 'gnomAD_EAS_AF', 'gnomAD_FIN_AF', 'gnomAD_NFE_AF', 'gnomAD_OTH_AF', 'gnomAD_SAS_AF', 'FILTER', 'Polyphen_Prediction', 

In [ ]:
# Define the feature variables (x)
model_features = ['Hugo_Symbol', 'Consequence', 'Variant_Type', 'gnomAD_AF', 'has_gnomAD_AF', 'Polyphen_Score', 'SIFT_Score', 'In_Hotspot', 'IN_DOMAIN', 'n_protein_domains', 'IN_FUNC_SITE', 'n_func_sites']

## Create ML dataframe 

In [ ]:
# Define the target variable (y) 
target = 'ONCOGENIC'
# Create df 
df_ml = variants[model_features + [target]].copy() 

In [ ]:
print(f"The dataset was reduced from {variants.shape[1]} to {df_ml.shape[1]} columns.")
print(f"Preview of the dataset:")
display(df_ml.head(10))

The dataset was reduced from 115 to 13 columns.
Preview of the dataset:


,Hugo_Symbol,Consequence,Variant_Type,gnomAD_AF,has_gnomAD_AF,Polyphen_Score,SIFT_Score,In_Hotspot,IN_DOMAIN,n_protein_domains,IN_FUNC_SITE,n_func_sites,ONCOGENIC
0,JAK1,missense_variant,SNP,NaN,False,0.992,0.01,False,True,1,False,0,Unknown
1,MAPK1,missense_variant,SNP,NaN,False,1.000,0.00,False,True,1,False,0,Resistance
2,MAPK1,missense_variant,SNP,NaN,False,0.640,0.00,False,True,1,True,1,Resistance
3,JAK1,missense_variant,SNP,NaN,False,0.996,0.00,False,True,1,False,0,Unknown
4,JAK1,missense_variant,SNP,NaN,False,0.999,0.00,False,True,1,False,0,Unknown
5,JAK1,stop_gained,SNP,NaN,False,NaN,NaN,False,True,1,False,0,Likely Oncogenic
6,MAPK1,missense_variant,SNP,NaN,False,0.954,0.00,False,True,1,True,1,Unknown
7,MAPK1,missense_variant,SNP,NaN,False,0.158,0.00,False,True,1,True,1,Resistance
8,MAPK1,missense_variant,SNP,NaN,False,0.997,0.00,False,True,1,False,0,Resistance
9,MAPK1,missense_variant,SNP,NaN,False,1.000,0.00,False,True,1,False,0,Resistance


## Filter to wanted oncogenic classes 

In [ ]:
wanted_classes = ['Oncogenic', 'Likely Neutral']
df_ml = df_ml[df_ml['ONCOGENIC'].isin(wanted_classes)]

print(f"After filtering the dataset contains {len(df_ml)} variants.")

print("Number of variants in each class:")
print(df_ml.groupby('ONCOGENIC').size())

After filtering the dataset contains 1906 variants.
Number of variants in each class:
ONCOGENIC
Likely Neutral     857
Oncogenic         1049
dtype: int64


## Check missing values 

In [ ]:
print("Number of rows with missing values, per feature:")
print(df_ml.isnull().sum())

Number of rows with missing values, per feature:
Hugo_Symbol             0
Consequence             0
Variant_Type            0
gnomAD_AF            1583
has_gnomAD_AF           0
Polyphen_Score        396
SIFT_Score            396
In_Hotspot              0
IN_DOMAIN               0
n_protein_domains       0
IN_FUNC_SITE            0
n_func_sites            0
ONCOGENIC               0
dtype: int64


## Create booleans for missing SIFT- and Polyphen scores

In [ ]:
# create boolean columns missing_polyphen and missing_sift 
# 1 = True, 0 = False
df_ml['missing_Polyphen'] = df_ml['Polyphen_Score'].isna().astype(int)
df_ml['missing_SIFT'] = df_ml['SIFT_Score'].isna().astype(int)

# check result
print(df_ml[['Polyphen_Score', 'missing_Polyphen', 'SIFT_Score', 'missing_SIFT']].head())

     Polyphen_Score  missing_Polyphen  SIFT_Score  missing_SIFT
337           1.000                 0        0.01             0
457           0.344                 0        0.00             0
458           0.026                 0        0.00             0
493           0.396                 0        0.12             0
495           0.838                 0        0.02             0


# Handle missing gnomAD AF values

In [ ]:
# fill missing values in 'gnomAD_AF' column with zero 
df_ml['gnomAD_AF'] = df_ml['gnomAD_AF'].fillna(0) 

# check for missing values 
print("Missing values check after handling:")
print(df_ml.isnull().sum())

Missing values check after handling:
Hugo_Symbol            0
Consequence            0
Variant_Type           0
gnomAD_AF              0
has_gnomAD_AF          0
Polyphen_Score       396
SIFT_Score           396
In_Hotspot             0
IN_DOMAIN              0
n_protein_domains      0
IN_FUNC_SITE           0
n_func_sites           0
ONCOGENIC              0
missing_Polyphen       0
missing_SIFT           0
dtype: int64


## Save ML dataframe 

In [ ]:
df_ml.to_csv("output/processed_variants_for_EDA.tsv", sep="\t", index=False) 